# Análisis batch del método dinámico para módulo de Young

Este notebook toma todos los CSV de mediciones dinámicas, permite analizarlos de a uno o en lote, y muestra gráficos de diagnóstico para ver dónde falla cada paso.

La cuenta usada para el módulo de Young es:

\[
\omega_0 = \sqrt{(2\pi f_1)^2 + \alpha^2}
\]

\[
E = \frac{64 m L^4 \omega_0^2}{\pi d^4 L_{\mathrm{total}} \beta_1^4}
\]

El análisis de \(\alpha\) usa solamente el método de máximos que venías usando: se detectan los picos superiores y se ajusta una exponencial.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.signal import find_peaks, detrend
from scipy.optimize import curve_fit
from IPython.display import display

# -------------------------------------------------------------------
# Carpeta de datos
# -------------------------------------------------------------------
# En tu PC, lo más cómodo es poner este notebook en la misma carpeta que los CSV.
# Si no, descomentá y editá la línea de Windows:
# CARPETA_DATOS = Path(r"C:\Users\user\Desktop\Labo 4\Young\dinamico")

CARPETA_DATOS = Path("/mnt/data") if Path("/mnt/data").exists() else Path.cwd()

ARCHIVOS_CSV = sorted(CARPETA_DATOS.glob("measurement_20260611_*.csv"))
print(f"Carpeta de datos: {CARPETA_DATOS}")
print(f"CSV encontrados: {len(ARCHIVOS_CSV)}")
for p in ARCHIVOS_CSV:
    print(" -", p.name)


In [ ]:

# -------------------------------------------------------------------
# Parámetros físicos
# -------------------------------------------------------------------
# No inventé datos para acero/cobre: completalos si querés que calcule E.
# Para latón dejé los valores que venías usando.

PARAMETROS_MATERIAL = {
    "laton": {
        "d": 5.00e-3,
        "err_d": 0.05e-3,
        "L_total": 50.0e-2,
        "err_L_total": 0.1e-2,
        "m": 82.76e-3,
        "err_m": 0.01e-3,
        "beta1": 1.875104,
    },
    "acero": {
        "d": np.nan,
        "err_d": np.nan,
        "L_total": np.nan,
        "err_L_total": np.nan,
        "m": np.nan,
        "err_m": np.nan,
        "beta1": 1.875104,
    },
    "cobre": {
        "d": np.nan,
        "err_d": np.nan,
        "L_total": np.nan,
        "err_L_total": np.nan,
        "m": np.nan,
        "err_m": np.nan,
        "beta1": 1.875104,
    },
}

# -------------------------------------------------------------------
# Configuración por archivo
# -------------------------------------------------------------------
# tmin/tmax: recorte temporal donde realmente querés analizar la oscilación.
# L: distancia efectiva desde el soporte hasta el filo/navaja, en metros.
# prominence y distance_s: parámetros de find_peaks.
# fmin/fmax: ventana donde buscar la frecuencia dominante en la FFT.

MEDICIONES = {
    "acero_1a": {
        "archivo": "measurement_20260611_113943.csv",
        "material": "acero",
        "L": 37e-2,
        "err_L": 1e-2,
        "tmin": 1.65,
        "tmax": 3.00,
        "prominence": 0.002,
        "distance_s": 0.008,
        "fmin": 1,
        "fmax": 150,
        "peak_kind": "max",
    },
    "acero_1b": {
        "archivo": "measurement_20260611_114030.csv",
        "material": "acero",
        "L": 37e-2,
        "err_L": 1e-2,
        "tmin": 1.65,
        "tmax": 3.00,
        "prominence": 0.002,
        "distance_s": 0.008,
        "fmin": 1,
        "fmax": 150,
        "peak_kind": "max",
    },
    "acero_2": {
        "archivo": "measurement_20260611_120130.csv",
        "material": "acero",
        "L": 37e-2,
        "err_L": 1e-2,
        "tmin": 1.55,
        "tmax": 3.50,
        "prominence": 0.03,
        "distance_s": 0.008,
        "fmin": 1,
        "fmax": 150,
        "peak_kind": "max",
    },
    "cobre_1": {
        "archivo": "measurement_20260611_121933.csv",
        "material": "cobre",
        "L": 31e-2,
        "err_L": 1e-2,
        "tmin": 8.45,
        "tmax": 9.95,
        "prominence": 0.005,
        "distance_s": 0.008,
        "fmin": 1,
        "fmax": 200,
        "peak_kind": "max",
    },
    "laton_1": {
        "archivo": "measurement_20260611_123917.csv",
        "material": "laton",
        "L": 29.5e-2,
        "err_L": 0.1e-2,
        "tmin": 1.20,
        "tmax": 6.20,
        "prominence": 0.03,
        "distance_s": 0.008,
        "fmin": 1,
        "fmax": 100,
        "peak_kind": "max",
    },
    "laton_2": {
        "archivo": "measurement_20260611_125450.csv",
        "material": "laton",
        "L": 29.5e-2,
        "err_L": 0.1e-2,
        "tmin": 0.50,
        "tmax": 2.40,
        "prominence": 0.01,
        "distance_s": 0.008,
        "fmin": 1,
        "fmax": 100,
        "peak_kind": "max",
    },
}

pd.DataFrame(MEDICIONES).T


In [ ]:

def modelo_exp(t, A, alpha, C):
    return A * np.exp(-alpha * t) + C


def leer_medicion(ruta):
    ruta = Path(ruta)
    with open(ruta, "r", encoding="utf-8", errors="replace") as f:
        comentario = f.readline().strip()

    df = pd.read_csv(ruta, skiprows=1)
    df = df.rename(columns={"Time (s)": "tiempo", "Voltage (V)": "voltaje"})
    df = df.dropna(subset=["tiempo", "voltaje"]).copy()
    return df, comentario


def recortar(df, tmin, tmax):
    mask = (df["tiempo"] >= tmin) & (df["tiempo"] <= tmax)
    out = df.loc[mask].copy()
    if len(out) < 10:
        raise ValueError(f"El recorte [{tmin}, {tmax}] s dejó menos de 10 puntos.")
    out["tiempo_recortado"] = out["tiempo"] - out["tiempo"].iloc[0]
    return out


def detectar_picos(t, v, prominence=0.01, distance_s=0.008):
    dt = np.median(np.diff(t))
    distance = max(1, int(round(distance_s / dt)))

    peaks_max, props_max = find_peaks(v, prominence=prominence, distance=distance)
    peaks_min, props_min = find_peaks(-v, prominence=prominence, distance=distance)

    return {
        "peaks_max": peaks_max,
        "peaks_min": peaks_min,
        "props_max": props_max,
        "props_min": props_min,
        "distance_samples": distance,
    }


def calcular_fft(t, v, fmin=1, fmax=100, detrend_type="constant", usar_ventana=True):
    dt = np.median(np.diff(t))

    if detrend_type in ["constant", "linear"]:
        y = detrend(v, type=detrend_type)
    else:
        y = v - np.mean(v)

    if usar_ventana:
        y_fft = y * np.hanning(len(y))
    else:
        y_fft = y

    fft = np.fft.rfft(y_fft)
    freq = np.fft.rfftfreq(len(y_fft), d=dt)
    amp = 2 * np.abs(fft) / len(y_fft)

    mask = (freq >= fmin) & (freq <= fmax)
    if not np.any(mask):
        raise ValueError("La ventana de frecuencias no contiene bins de la FFT.")

    idxs = np.flatnonzero(mask)
    idx_dom = idxs[np.argmax(amp[mask])]

    f1 = freq[idx_dom]
    amp1 = amp[idx_dom]
    err_f1 = freq[1] - freq[0]

    return {
        "freq": freq,
        "amp": amp,
        "f1": f1,
        "amp1": amp1,
        "err_f1": err_f1,
        "idx_dom": idx_dom,
        "dt": dt,
    }


def ajustar_alpha_picos(t, v, peaks, peak_kind="max", min_picos=6):
    if peak_kind == "max":
        idx = peaks["peaks_max"]
        etiqueta = "máximos"
    elif peak_kind == "min":
        idx = peaks["peaks_min"]
        etiqueta = "mínimos"
    else:
        raise ValueError("peak_kind debe ser 'max' o 'min'.")

    if len(idx) < min_picos:
        raise ValueError(f"Hay muy pocos {etiqueta}: {len(idx)} detectados.")

    tp = t[idx]
    vp = v[idx]

    A0 = vp[0] - vp[-1]
    alpha0 = 1 / max(tp[-1] - tp[0], 1e-9)
    C0 = vp[-1]

    popt, pcov = curve_fit(
        modelo_exp,
        tp,
        vp,
        p0=[A0, alpha0, C0],
        maxfev=20000,
    )

    perr = np.sqrt(np.diag(pcov))
    residuos = vp - modelo_exp(tp, *popt)
    escala = np.ptp(vp) if np.ptp(vp) > 0 else np.std(vp)
    rmse_rel = np.sqrt(np.mean(residuos**2)) / escala

    return {
        "metodo_alpha": f"picos_{etiqueta}",
        "t_env": tp,
        "v_env": vp,
        "popt": popt,
        "perr": perr,
        "residuos": residuos,
        "rmse_rel": rmse_rel,
    }


def calcular_E(f1, err_f1, alpha, err_alpha, cfg, params_material):
    requeridos = ["d", "err_d", "L_total", "err_L_total", "m", "err_m", "beta1"]
    faltan = [k for k in requeridos if k not in params_material or not np.isfinite(params_material[k])]
    faltan += [k for k in ["L", "err_L"] if k not in cfg or not np.isfinite(cfg[k])]

    if faltan:
        return {
            "E": np.nan,
            "err_E": np.nan,
            "omega0": np.nan,
            "err_omega0": np.nan,
            "warning_E": "Faltan parámetros físicos para E: " + ", ".join(sorted(set(faltan))),
        }

    d = params_material["d"]
    err_d = params_material["err_d"]
    L_total = params_material["L_total"]
    err_L_total = params_material["err_L_total"]
    m = params_material["m"]
    err_m = params_material["err_m"]
    beta1 = params_material["beta1"]
    L = cfg["L"]
    err_L = cfg["err_L"]

    omega0 = np.sqrt((2 * np.pi * f1) ** 2 + alpha**2)

    domega0_df1 = (4 * np.pi**2 * f1) / omega0
    domega0_dalpha = alpha / omega0
    err_omega0 = np.sqrt((domega0_df1 * err_f1) ** 2 + (domega0_dalpha * err_alpha) ** 2)

    E = 64 * m * L**4 * omega0**2 / (np.pi * d**4 * L_total * beta1**4)

    Q = omega0**2
    err_Q = np.sqrt((8 * np.pi**2 * f1 * err_f1) ** 2 + (2 * alpha * err_alpha) ** 2)

    err_E = E * np.sqrt(
        (4 * err_d / d) ** 2
        + (err_L_total / L_total) ** 2
        + (err_m / m) ** 2
        + (4 * err_L / L) ** 2
        + (err_Q / Q) ** 2
    )

    return {
        "E": E,
        "err_E": err_E,
        "omega0": omega0,
        "err_omega0": err_omega0,
        "warning_E": "",
    }


def diagnosticar_resultado(alpha, err_alpha, rmse_rel, n_picos, metodo_alpha):
    warnings = []
    if not np.isfinite(alpha) or alpha <= 0:
        warnings.append("alpha no positivo o no finito")
    if np.isfinite(err_alpha) and np.isfinite(alpha) and alpha != 0 and abs(err_alpha / alpha) > 0.5:
        warnings.append("incertidumbre relativa de alpha muy grande")
    if np.isfinite(rmse_rel) and rmse_rel > 0.25:
        warnings.append("ajuste de envolvente pobre")
    if n_picos < 6 and metodo_alpha.startswith("picos"):
        warnings.append("muy pocos picos detectados")
    return warnings


In [ ]:
def analizar_uno(nombre, plot=True, guardar_figuras=False, carpeta_figuras="figuras_young"):
    """
    Analiza una medición.

    Parámetros útiles:
    - nombre: clave de MEDICIONES, por ejemplo "laton_2".
    - plot=True: muestra gráficos de diagnóstico.
    - guardar_figuras=True: además guarda las figuras en PNG.

    El ajuste de alpha se hace con el método original:
    detectar picos superiores y ajustar A exp(-alpha t) + C.
    """
    if nombre not in MEDICIONES:
        raise KeyError(f"No existe {nombre}. Opciones: {list(MEDICIONES)}")

    cfg = MEDICIONES[nombre].copy()
    ruta = CARPETA_DATOS / cfg["archivo"]
    if not ruta.exists():
        raise FileNotFoundError(f"No encontré el archivo: {ruta}")

    df, comentario = leer_medicion(ruta)
    df_rec = recortar(df, cfg["tmin"], cfg["tmax"])

    t = df_rec["tiempo_recortado"].to_numpy()
    v = df_rec["voltaje"].to_numpy()

    peaks = detectar_picos(t, v, cfg["prominence"], cfg["distance_s"])
    fft_res = calcular_fft(t, v, cfg["fmin"], cfg["fmax"], detrend_type="linear")

    # Único método de alpha: picos superiores, como venías haciendo.
    fit_alpha = ajustar_alpha_picos(t, v, peaks, peak_kind=cfg.get("peak_kind", "max"))

    A, alpha, C = fit_alpha["popt"]
    err_A, err_alpha, err_C = fit_alpha["perr"]

    params_mat = PARAMETROS_MATERIAL.get(cfg["material"], {})
    E_res = calcular_E(
        fft_res["f1"],
        fft_res["err_f1"],
        alpha,
        err_alpha,
        cfg,
        params_mat,
    )

    n_picos = len(peaks["peaks_max"])
    warnings = diagnosticar_resultado(alpha, err_alpha, fit_alpha["rmse_rel"], n_picos, fit_alpha["metodo_alpha"])
    if E_res["warning_E"]:
        warnings.append(E_res["warning_E"])

    resultado = {
        "nombre": nombre,
        "archivo": cfg["archivo"],
        "comentario": comentario,
        "material": cfg["material"],
        "tmin": cfg["tmin"],
        "tmax": cfg["tmax"],
        "L_m": cfg.get("L", np.nan),
        "n_puntos_recorte": len(df_rec),
        "n_maximos": len(peaks["peaks_max"]),
        "n_minimos": len(peaks["peaks_min"]),
        "metodo_alpha": fit_alpha["metodo_alpha"],
        "A_V": A,
        "err_A_V": err_A,
        "alpha_1_s": alpha,
        "err_alpha_1_s": err_alpha,
        "C_V": C,
        "err_C_V": err_C,
        "rmse_rel_env": fit_alpha["rmse_rel"],
        "f1_Hz": fft_res["f1"],
        "err_f1_Hz": fft_res["err_f1"],
        "omega0_rad_s": E_res["omega0"],
        "err_omega0_rad_s": E_res["err_omega0"],
        "E_Pa": E_res["E"],
        "err_E_Pa": E_res["err_E"],
        "E_GPa": E_res["E"] / 1e9 if np.isfinite(E_res["E"]) else np.nan,
        "err_E_GPa": E_res["err_E"] / 1e9 if np.isfinite(E_res["err_E"]) else np.nan,
        "warnings": "; ".join(warnings),
        "estado": "ok" if len(warnings) == 0 else "revisar",
    }

    figs = []
    if plot or guardar_figuras:
        if guardar_figuras:
            carpeta = Path(carpeta_figuras)
            carpeta.mkdir(parents=True, exist_ok=True)

        # 1. Señal completa y recorte
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(df["tiempo"], df["voltaje"], lw=0.7)
        ax.axvspan(cfg["tmin"], cfg["tmax"], alpha=0.2, label="recorte usado")
        ax.set_xlabel("Tiempo (s)")
        ax.set_ylabel("Voltaje (V)")
        ax.set_title(f"{nombre}: señal completa")
        ax.grid(True)
        ax.legend()
        figs.append(("senal_completa", fig))

        # 2. Recorte y picos
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(t, v, lw=0.7, label="señal recortada")
        ax.scatter(t[peaks["peaks_max"]], v[peaks["peaks_max"]], s=18, label="máximos")
        ax.scatter(t[peaks["peaks_min"]], v[peaks["peaks_min"]], s=18, label="mínimos")
        ax.set_xlabel("Tiempo desde el recorte (s)")
        ax.set_ylabel("Voltaje (V)")
        ax.set_title(f"{nombre}: picos detectados")
        ax.grid(True)
        ax.legend()
        figs.append(("picos", fig))

        # 3. Ajuste de envolvente
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
        ax1.plot(t, v, lw=0.5, alpha=0.6, label="señal")

        te = fit_alpha["t_env"]
        ve = fit_alpha["v_env"]
        ax1.scatter(te, ve, s=14, label="picos usados para el ajuste")

        t_fit = np.linspace(te.min(), te.max(), 1200)
        ax1.plot(t_fit, modelo_exp(t_fit, A, alpha, C), lw=2, label="ajuste exponencial")
        ax1.set_ylabel("Voltaje (V)")
        ax1.set_title(f"{nombre}: ajuste de alpha con picos superiores")
        ax1.grid(True)
        ax1.legend()

        ax2.scatter(te, fit_alpha["residuos"], s=10)
        ax2.axhline(0, lw=1)
        ax2.set_xlabel("Tiempo desde el recorte (s)")
        ax2.set_ylabel("Residuo")
        ax2.grid(True)
        figs.append(("ajuste_alpha", fig))

        # 4. FFT
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(fft_res["freq"], fft_res["amp"], lw=0.9)
        ax.axvline(fft_res["f1"], ls="--", label=f"f1 = {fft_res['f1']:.3g} Hz")
        ax.set_xlim(cfg["fmin"], cfg["fmax"])
        ax.set_xlabel("Frecuencia (Hz)")
        ax.set_ylabel("Amplitud")
        ax.set_title(f"{nombre}: FFT del recorte")
        ax.grid(True)
        ax.legend()
        figs.append(("fft", fig))

        for tag, fig in figs:
            fig.tight_layout()
            if guardar_figuras:
                fig.savefig(carpeta / f"{nombre}_{tag}.png", dpi=160)
            if plot:
                plt.show()
            else:
                plt.close(fig)

    return resultado


def analizar_todos(plot=True, guardar_figuras=True):
    """
    Corre todas las mediciones usando el método original de picos superiores.
    Si plot=True, muestra los cuatro gráficos de diagnóstico de cada medición.
    """
    resultados = []
    for nombre in MEDICIONES:
        print(f"\nAnalizando {nombre}...")
        try:
            res = analizar_uno(nombre, plot=plot, guardar_figuras=guardar_figuras)
        except Exception as e:
            res = {
                "nombre": nombre,
                "archivo": MEDICIONES[nombre].get("archivo", ""),
                "estado": "fallo",
                "warnings": repr(e),
            }
            print(f"Falló {nombre}: {e!r}")
        resultados.append(res)

    tabla = pd.DataFrame(resultados)
    cols_principales = [
        "nombre", "estado", "material", "metodo_alpha", "f1_Hz", "err_f1_Hz",
        "alpha_1_s", "err_alpha_1_s", "E_GPa", "err_E_GPa", "warnings",
    ]
    cols = [c for c in cols_principales if c in tabla.columns] + [c for c in tabla.columns if c not in cols_principales]
    return tabla[cols]


In [ ]:

def detectar_duplicados_por_contenido():
    """
    Detecta CSV que tienen exactamente los mismos datos numéricos.
    Ignora la línea de comentario.
    """
    firmas = {}
    duplicados = []

    for nombre, cfg in MEDICIONES.items():
        ruta = CARPETA_DATOS / cfg["archivo"]
        if not ruta.exists():
            continue
        df, _ = leer_medicion(ruta)
        firma = pd.util.hash_pandas_object(df, index=False).sum()
        if firma in firmas:
            duplicados.append((firmas[firma], nombre))
        else:
            firmas[firma] = nombre

    return duplicados


detectar_duplicados_por_contenido()


## Uso 1: analizar una medición sola

Esto es lo más útil cuando algo falla. Cambiá el nombre por cualquiera de las claves de `MEDICIONES`.


In [ ]:
res = analizar_uno("laton_2", plot=True, guardar_figuras=False)
pd.Series(res)


## Uso 2: correr todas y mostrar todos los gráficos

Esta celda corre todas las mediciones con el mismo método que venías usando. Para cada archivo muestra: señal completa con recorte, picos detectados, ajuste exponencial y FFT.


In [ ]:
tabla_picos = analizar_todos(plot=True, guardar_figuras=True)
display(tabla_picos)



## Comparación rápida de resultados disponibles

El gráfico ignora automáticamente las mediciones donde faltan parámetros físicos para calcular \(E\).


In [ ]:

def graficar_E(tabla):
    t = tabla.copy()
    t = t[np.isfinite(t.get("E_GPa", np.nan))]
    if len(t) == 0:
        print("No hay valores de E para graficar. Completá PARAMETROS_MATERIAL para acero/cobre si hace falta.")
        return

    plt.figure(figsize=(8, 4))
    plt.errorbar(t["nombre"], t["E_GPa"], yerr=t["err_E_GPa"], fmt="o", capsize=4)
    plt.ylabel("E (GPa)")
    plt.xlabel("Medición")
    plt.title("Módulo de Young por medición")
    plt.grid(True)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

# Elegí qué tabla querés graficar:
graficar_E(tabla_picos)
